In [1]:
# Install OpenAI API library (used by NVIDIA NIM)
%pip install -U openai

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 9.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports and Inititalizations

In [5]:
import pandas as pd
import base64
import os
from openai import OpenAI
from io import BytesIO
from PIL import Image
import nltk
import json
import itertools

env_path = ".env" 
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        API_KEYS = [line.strip() for line in f if line.strip() and not line.startswith("#")]
else:
    print(f"Error: '{env_path}' file not found. Ensure it is in: {os.getcwd()}")
    API_KEYS = []

# Create a generator that cycles through the keys infinitely
if API_KEYS:
    key_cycle = itertools.cycle(API_KEYS)
    print(f"Successfully loaded {len(API_KEYS)} API keys from .env for rotation.")
else:
    key_cycle = None
    print("Warning: No API keys loaded. The inference engine will fail.")

# Evaluator used in CC_OCR
from doc_parsing_evaluator import ParsingEvaluator

# Ensure NLTK data is available
nltk.download('punkt')

# Paths based on your local directory structure
BASE_DIR = r"C:\Users\anshg\OneDrive\Desktop\BTP\Evaluation-Of-MultiModal-LLMs-for-Layout-Aware-Document-Parsing\CC-OCR_Dataset\doc_parsing"
OUT_DIR = r"C:\Users\anshg\OneDrive\Desktop\BTP\Evaluation-Of-MultiModal-LLMs-for-Layout-Aware-Document-Parsing\Evaluation_Results"
os.makedirs(OUT_DIR, exist_ok=True)

# IMPORTANT: Update this to a multimodal model available on NVIDIA NIM 
# Example: "meta/llama-3.2-90b-vision-instruct" or whatever you are testing
MODEL_NAME = "meta/llama-4-maverick-17b-128e-instruct" 

# Initialize the evaluator
evaluator = ParsingEvaluator(group_name="OCR_Eval")

Successfully loaded 1 API keys from .env for rotation.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\anshg\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Inference Engine (Nvidea NIM APIs)

In [6]:
def query_nim(image_bytes, task_type):
    """Sends image to NVIDIA NIM API, rotating through available API keys"""
    
    if key_cycle is None:
        return ""
        
    # Pull the next key and initialize the NVIDIA NIM client
    current_key = next(key_cycle)
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=current_key
    )
    
    if task_type == "table":
        prompt = "Parse this table into a clean, structured HTML format with <table> tags. Include all row and column spans."
    else:
        prompt = "Parse this document into LaTeX format. Provide only the LaTeX code."

    # Convert bytes back to base64 for the API payload
    image_b64 = base64.b64encode(image_bytes).decode("utf-8")
    
    # Generate content using the current key
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/png;base64,{image_b64}"}
                        }
                    ]
                }
            ],
            max_tokens=2048,
            temperature=0.2 # Recommended low temp for strict parsing tasks
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error with key {current_key[:10]}...: {e}")
        return ""

## Document Parsing Evaluation (LaTeX)

In [3]:
doc_tasks = {
    "Scanned": os.path.join(BASE_DIR, "doc", "doc_scan_eng_75.tsv"),
    "Photoed": os.path.join(BASE_DIR, "doc", "doc_photo_eng_75.tsv")
}

doc_results = {}

for label, path in doc_tasks.items():
    if not os.path.exists(path): continue
    df = pd.read_csv(path, sep='\t')
    scores = []
    
    print(f"Processing {label} documents...")
    
    for _, row in df.iterrows():
        img_bytes = base64.b64decode(row['image'])
        ground_truth = str(row['answer']).strip() 
        
        prediction = query_nim(img_bytes, "doc")
        
        # FIX: Use the professional NED logic with LaTeX cleaning
        score = evaluator.evaluate_single_doc_sample(ground_truth, prediction)
        scores.append(score)
        
    doc_results[label] = sum(scores) / len(scores) if scores else 0

print(f"Document Parsing (NED with LaTeX) Results: {doc_results}")

Processing Scanned documents...
Processing Photoed documents...
Document Parsing (NED with LaTeX) Results: {'Scanned': 0.7132478665517284, 'Photoed': 0.715749670088711}


## Table Parsing Evaluation (HTML)

In [7]:
import os
from tqdm import tqdm # Make sure this is imported at the top of your file


table_tasks = {
    "Scanned": os.path.join(BASE_DIR, "table", "table_scan_eng_75.tsv"),
    "Photoed": os.path.join(BASE_DIR, "table", "table_photo_eng_75.tsv")
}

table_results = {}

for label, path in table_tasks.items():
    if not os.path.exists(path): 
        print(f"Skipping {label}: Path not found.")
        continue
        
    df = pd.read_csv(path, sep='\t')
    scores = []
    
    print(f"Processing {label} tables...")
    
    # Wrapped the iterator in tqdm for the progress bar
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Scoring {label}"):
        img_bytes = base64.b64decode(row['image'])
        ground_truth_html = str(row['answer']).strip() 
        
        prediction = query_nim(img_bytes, "table")
        
        # FIX: Use the official Tree Edit Distance (TEDS) metric
        score = evaluator.evaluate_single_table_sample(ground_truth_html, prediction)
        scores.append(score)
        
    table_results[label] = sum(scores) / len(scores) if scores else 0

print(f"\nTable Parsing (TEDS Results): {table_results}")

Processing Scanned tables...


Scoring Scanned: 100%|██████████| 75/75 [3:05:10<00:00, 148.15s/it]    


Processing Photoed tables...


Scoring Photoed: 100%|██████████| 75/75 [13:56<00:00, 11.15s/it]


Table Parsing (TEDS Results): {'Scanned': 0.6990777780714308, 'Photoed': 0.566983886961906}


## Final Report 

In [ ]:
summary = {
    "Document Parsing (NED-LaTeX)": doc_results,
    "Table Parsing (HTML-Sim)": table_results,
    "Model": MODEL_NAME,
    "Track": "English Document Parsing"
}

with open(os.path.join(OUT_DIR, "stage1_final_report.json"), "w") as f:
    json.dump(summary, f, indent=4)

print("Evaluation Complete. Check 'stage1_final_report.json' for full details.")

Evaluation Complete. Check 'stage1_final_report.json' for full details.
